# Lesson 5.4 — CI/CD Integration

**Phase 5: Real-World Workflows**

---

## Learning Objectives

By the end of this lesson you will be able to:
- Design a complete Helm CI/CD pipeline
- Write GitHub Actions and GitLab CI workflows for Helm
- Use `helm upgrade --install` for idempotent deployments
- Implement environment-specific value overrides with files
- Understand Helm in GitOps workflows (Flux, ArgoCD)

---

## The Helm CI/CD Pipeline

A typical Helm pipeline has two stages:

```
CI (Pull Request):
  helm lint
  helm template | kubectl apply --dry-run
  helm test (against a test cluster)

CD (Merge to main / tag):
  helm package
  helm push → OCI registry
  helm upgrade --install → target environment
  helm test (smoke test deployed release)
```

---

## `helm upgrade --install` — The Idempotent Deploy

In CI/CD, always use `--install` flag with `upgrade`. It installs if not present, upgrades if it is:

In [ ]:
echo "The idempotent deploy command pattern:"
echo ""
cat << 'EOF'
helm upgrade --install my-app oci://registry/my-app \
  --version 1.2.3 \
  --namespace production \
  --create-namespace \
  --values values-base.yaml \
  --values values-production.yaml \
  --set image.tag=${COMMIT_SHA} \
  --wait \
  --timeout 5m \
  --atomic
EOF

echo ""
echo "Key flags:"
echo "  --install      Install if not deployed, upgrade if already deployed"
echo "  --wait         Wait until all resources are Ready"
echo "  --atomic       Roll back automatically if deployment fails"
echo "  --timeout      Max time to wait (default 5m)"
echo "  --create-namespace  Create namespace if it doesn't exist"

---

## Environment-Specific Values Files

The recommended pattern is to maintain layered values files:

In [ ]:
# Create environment-specific values files
mkdir -p /tmp/envvalues

cat > /tmp/envvalues/values-base.yaml << 'EOF'
# Shared defaults for all environments
image:
  repository: myregistry/myapp
  tag: ""  # Set by CI pipeline via --set image.tag=$COMMIT_SHA

resources:
  requests:
    cpu: 100m
    memory: 128Mi
  limits:
    cpu: 500m
    memory: 512Mi
EOF

cat > /tmp/envvalues/values-staging.yaml << 'EOF'
# Staging-specific overrides
replicaCount: 1
service:
  type: ClusterIP
ingress:
  enabled: true
  host: myapp.staging.example.com
EOF

cat > /tmp/envvalues/values-production.yaml << 'EOF'
# Production-specific overrides
replicaCount: 3
service:
  type: LoadBalancer
ingress:
  enabled: true
  host: myapp.example.com
  tls: true
resources:
  requests:
    cpu: 200m
    memory: 256Mi
  limits:
    cpu: 1000m
    memory: 1Gi
EOF

echo "Values files created:"
ls /tmp/envvalues/

---

## GitHub Actions Pipeline

In [ ]:
%%writefile /tmp/helm-ci-github-actions.yaml
# .github/workflows/helm-deploy.yml
name: Helm CI/CD

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

env:
  REGISTRY: ghcr.io
  CHART_REPO: ghcr.io/${{ github.repository_owner }}/charts

jobs:
  lint-and-test:
    name: Lint & Validate
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - name: Set up Helm
        uses: azure/setup-helm@v3
        with:
          version: v4.x.x

      - name: Helm lint
        run: helm lint ./chart --strict

      - name: Helm template (render check)
        run: |
          helm template my-app ./chart \
            -f ./chart/values-staging.yaml \
            --set image.tag=${{ github.sha }}

      - name: Kubectl dry-run
        run: |
          helm template my-app ./chart \
            -f ./chart/values-staging.yaml \
            --set image.tag=${{ github.sha }} \
            | kubectl apply --dry-run=client -f -
        # Requires kubectl configured against a test cluster or use kind action

  package-and-push:
    name: Package & Publish Chart
    runs-on: ubuntu-latest
    needs: lint-and-test
    if: github.ref == 'refs/heads/main'
    steps:
      - uses: actions/checkout@v4

      - name: Set up Helm
        uses: azure/setup-helm@v3

      - name: Log in to OCI Registry
        run: |
          echo ${{ secrets.GITHUB_TOKEN }} | helm registry login ghcr.io \
            -u ${{ github.actor }} --password-stdin

      - name: Package chart
        run: helm package ./chart --destination /tmp/packaged

      - name: Push chart to OCI registry
        run: |
          helm push /tmp/packaged/*.tgz oci://${{ env.CHART_REPO }}

  deploy-staging:
    name: Deploy to Staging
    runs-on: ubuntu-latest
    needs: package-and-push
    environment: staging
    steps:
      - uses: actions/checkout@v4

      - name: Deploy to Staging
        run: |
          CHART_VERSION=$(helm show chart oci://${{ env.CHART_REPO }}/myapp 2>/dev/null | grep '^version:' | awk '{print $2}')
          helm upgrade --install my-app \
            oci://${{ env.CHART_REPO }}/myapp \
            --version ${CHART_VERSION} \
            --namespace staging \
            --create-namespace \
            --values ./chart/values-staging.yaml \
            --set image.tag=${{ github.sha }} \
            --wait --atomic --timeout 5m

      - name: Run Helm tests
        run: helm test my-app --namespace staging

---

## GitLab CI Pipeline

In [ ]:
%%writefile /tmp/helm-ci-gitlab.yaml
# .gitlab-ci.yml
stages:
  - validate
  - package
  - deploy

variables:
  HELM_VERSION: "4.x.x"
  CHART_PATH: ./chart
  REGISTRY: registry.gitlab.com/$CI_PROJECT_PATH

default:
  image:
    name: alpine/helm:${HELM_VERSION}
    entrypoint: [""]

helm-lint:
  stage: validate
  script:
    - helm lint ${CHART_PATH} --strict
    - helm lint ${CHART_PATH} -f ${CHART_PATH}/values-staging.yaml

helm-template:
  stage: validate
  script:
    - helm template test-release ${CHART_PATH}
      -f ${CHART_PATH}/values-staging.yaml
      --set image.tag=${CI_COMMIT_SHA}
      > /dev/null
    - echo "Template rendering successful"

helm-package-push:
  stage: package
  only:
    - main
  script:
    - echo ${CI_REGISTRY_PASSWORD} | helm registry login ${CI_REGISTRY} -u ${CI_REGISTRY_USER} --password-stdin
    - helm package ${CHART_PATH} --destination /tmp
    - helm push /tmp/*.tgz oci://${REGISTRY}/charts

deploy-staging:
  stage: deploy
  environment:
    name: staging
    url: https://myapp.staging.example.com
  only:
    - main
  script:
    - helm upgrade --install my-app
        oci://${REGISTRY}/charts/myapp
        --namespace staging
        --create-namespace
        -f ${CHART_PATH}/values-staging.yaml
        --set image.tag=${CI_COMMIT_SHA}
        --wait --atomic --timeout 5m

---

## Helm in GitOps (Flux & ArgoCD)

In GitOps workflows, a controller (Flux or ArgoCD) continuously reconciles the cluster to match a Git repository. Both support Helm natively:

In [ ]:
# Flux HelmRelease CRD example
cat << 'EOF'
# flux-helmrelease.yaml — committed to Git, watched by Flux
apiVersion: helm.toolkit.fluxcd.io/v2beta2
kind: HelmRelease
metadata:
  name: my-app
  namespace: production
spec:
  interval: 10m            # Reconcile every 10 minutes
  chart:
    spec:
      chart: my-app
      version: "1.x.x"    # Semver constraint
      sourceRef:
        kind: HelmRepository
        name: my-oci-registry
  values:
    replicaCount: 3
    image:
      tag: "1.2.3"
  valuesFrom:
    - kind: Secret
      name: my-app-secrets  # Reference a Kubernetes Secret for sensitive values
EOF

In [ ]:
# ArgoCD Application using a Helm chart
cat << 'EOF'
# argocd-application.yaml
apiVersion: argoproj.io/v1alpha1
kind: Application
metadata:
  name: my-app
  namespace: argocd
spec:
  project: default
  source:
    repoURL: oci://ghcr.io/myorg/charts
    chart: my-app
    targetRevision: 1.2.3
    helm:
      valueFiles:
        - values-production.yaml
      parameters:
        - name: image.tag
          value: abc123def456  # Override a single value
  destination:
    server: https://kubernetes.default.svc
    namespace: production
  syncPolicy:
    automated:
      prune: true
      selfHeal: true
    syncOptions:
      - CreateNamespace=true
EOF

---

## Exercise

In [ ]:
# TASK 1: Simulate the CI lint+render pipeline for the lintdemo chart:
#   Step 1: helm lint
#   Step 2: helm template (render to stdout)
#   Step 3 (if kubectl available): pipe to kubectl apply --dry-run=client
# TODO:
CHART=/tmp/lintdemo
# helm lint ${CHART}
# helm template test-release ${CHART} | head -20
# helm template test-release ${CHART} | kubectl apply --dry-run=client -f -


# TASK 2: Create values files for 3 environments (dev, staging, prod)
#         with different replicaCount and resource values.
#         Render the lintdemo chart with each and compare the output.
# TODO:


# TASK 3: Write a minimal shell script 'deploy.sh' that:
#   - Takes $ENVIRONMENT and $IMAGE_TAG as arguments
#   - Runs helm upgrade --install with the matching values file
#   - Includes --atomic for automatic rollback on failure
# TODO:

echo "Exercise: implement the tasks above"

---

## Summary

- Use `helm upgrade --install --atomic --wait` for idempotent, safe deployments in CI/CD.
- Layer values files: `values-base.yaml` → `values-<env>.yaml` → `--set image.tag=<commit>`.
- CI pipeline: lint → template → dry-run → package → push → deploy → test.
- GitOps tools (Flux, ArgoCD) can manage Helm releases declaratively from Git.
- Always use `--atomic` in production pipelines — it rolls back automatically on failure.

---

## Next Step

You've completed all 5 phases! Time to apply everything in the capstone project.

➡️ [Capstone Project — Deploy a Multi-Service App](../capstone/capstone-project.ipynb)